# **TELECOM CUSTOMER CHURN PREDICTION**

### Customer Churn

Customer churn occurs when customers stop using a company’s service. In the telecom industry, customers frequently switch providers, resulting in high annual churn rates of 15–25%. Since retaining existing customers costs less than acquiring new ones, companies focus on predicting which customers are likely to leave. By analyzing customer behavior and interactions, businesses can target high-risk customers with effective retention strategies, helping reduce losses and increase profitability.

### Objectives
I will explore the data and try to answer some questions like:

- What's the % of Churn Customers and customers that keep in with the active services?
- Is there any patterns in Churn Customers based on the gender?
- Is there any patterns/preference in Churn Customers based on the type of service provided?
- What's the most profitable service types?
- Which features and services are most profitable?
- Many more questions that will arise during the analysis

## Importing Libraries and Importing Dataset

### Importing Libraries

Loading all necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

### Importing Dataset

Importing the dataset from Kaggle and converting it into a pandas DataFrame for analysis and preprocessing.

## ***For now using csv because of incorrrect data importing***

In [ ]:
df = pd.read_csv('telco_dataset.csv')

# Exploratory Data Analysis (EDA)

In EDA, we will explore and understand the dataset by analyzing patterns, distributions, relationships, and detecting any missing or unusual values before building models.

In [ ]:
# Loading top 5 rows
df.head()

In [ ]:
# Loading bottom 5 rows
df.tail()

The data set includes information about:

- Customers who left within the last month – the column is called Churn

- Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies

- Customer account information - how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges

- Demographic info about customers – gender, age range, and if they have partners and dependents

In [ ]:
# Dimention of Dataset
df.shape

In [ ]:
# Info about the dataset
df.info()

In [ ]:
# Array of all columns in Dataset
df.columns.values

In [ ]:
# Showing all column Types
df.dtypes

-> The target the we will use to guide the exploration is Churn

## Visualize missing values

In [ ]:
# Visualize missing values as a matrix
msno.matrix(df);

Using this matrix we can very quickly find the pattern of missingness in the dataset.

- From the above visualisation we can observe that it has no peculiar pattern that stands out. In fact there is no missing data.

## Data Manipulation

In [ ]:
# Removing Unnecessary Columns, which is unnecessary for model training
df = df.drop(['customerID'], axis = 1)
df.head()

-> On deep analysis, we can find some indirect missingness in our data (which can be in form of blankspaces)

In [ ]:
# Convert TotalCharges to numeric (invalid values become NaN) and check missing values in each column
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()

Here we see that the TotalCharges has 11 missing values. Let's check this data.

In [ ]:
# Showing rows where TotalCharges is NaN (missing)
df[np.isnan(df['TotalCharges'])]

It can also be noted that the Tenure column is 0 for these entries even though the MonthlyCharges column is not empty.

In [ ]:
# Get index numbers of rows where tenure is 0
df[df['tenure'] == 0].index

There are no additional missing values in the Tenure column

In [ ]:
# Remove rows where tenure is 0 and verify they are deleted
df.drop(labels=df[df['tenure'] == 0].index, axis=0, inplace=True)
df[df['tenure'] == 0].index

In [ ]:
df.fillna(df["TotalCharges"].mean())

In [ ]:
df.isnull().sum()

In [ ]:
df['Churn'].value_counts()

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
# Select numerical columns
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Plot boxplots for each numerical feature vs Churn
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Churn', y=col, data=df)
    plt.title(f"{col} vs Churn")
    plt.show()

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[['tenure','MonthlyCharges','TotalCharges','Churn']].corr(),
            annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix")m
plt.show()

## Encoding Categorical Variables
Machine learning models cannot work with categorical (text) data.
Therefore, we convert all categorical features into numerical format
using One-Hot Encoding.

We use drop_first=True to avoid multicollinearity
(dummy variable trap).

In [ ]:
# Check categorical columns
df.select_dtypes(include='object').columns

In [ ]:
# Apply one-hot encoding
df = pd.get_dummies(df, drop_first=True)

df.head()

## Splitting Features and Target Variable
In this step, we separate the independent variables (X)
from the dependent variable (y).

X → All input features  
y → Target variable (Churn)

This separation is necessary before training a machine learning model.

In [ ]:
# Separating features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Checking shapes
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)